<div style="display: flex; align-items: center;">
    <h1>Optimizing parameters in a WOFOST crop model using <code>diffWOFOST</code></h1>
    <img src="https://raw.githubusercontent.com/WUR-AI/diffWOFOST/refs/heads/main/docs/logo/diffwofost.png" width="150" style="margin-left: 20px;">
</div>


This Jupyter notebook demonstrates the optimization of parameters in a
differentiable model using the `diffwofost` package. The package provides
differentiable implementations of the WOFOST model and its associated
sub-models. As `diffwofost` is under active development, this notebook focuses on
one model: `wofost72_pp`. 

## 1. WOFOST72_PP

In this section, we will demonstrate how to optimize the parameters `TDWI`, `SPAN` in
`wofost72_pp` model using a differentiable version of wofost.
The optimization will be done using the Adam optimizer from `torch.optim`.

### 1.1 software requirements

To run this notebook, we need to install the `diffwofost`; the differentiable
version of WOFOST models. Since the package is constantly under development, make
sure you have the latest version of `diffwofost` installed in your
python environment. You can install it using pip:

In [ ]:
# install diffwofost
!pip install diffwofost

In [1]:
# ---- import libraries ----
import copy
import torch
import numpy
from pathlib import Path
from diffwofost.physical_models.config import Configuration, ComputeConfig
from diffwofost.physical_models.crop.wofost72 import Wofost72
from diffwofost.physical_models.soil.classic_waterbalance import WaterbalancePP
from diffwofost.physical_models.utils import EngineTestHelper
from diffwofost.physical_models.utils import prepare_engine_input
from diffwofost.physical_models.utils import get_test_data

In [2]:
# ---- disable a warning: this will be fixed in the future ----
import warnings
warnings.filterwarnings("ignore", message="To copy construct from a tensor.*")

### 1.2. Data

A test dataset of `LAI` (Leaf Area Index) will be used to optimize the parameters:
- `TWDI`: total initial dry weight
- `SPAN`: life span of leaves

The data is stored in the PCSE tests folder, and can be downloaded from the PCSE repository.
You can select any of the files related to `potentialproduction_wofost72` model with a file name that follows the pattern
`test_potentialproduction_wofost72_*.yaml`. Each file contains different data depending on the location and crop type.
For example, you can download the file "test_potentialproduction_wofost72_03" as:

In [3]:
import urllib.request

filename = "test_potentialproduction_wofost72_03.yaml"
url = f"https://raw.githubusercontent.com/ajwdewit/pcse/refs/heads/master/tests/test_data/{filename}"

urllib.request.urlretrieve(url, filename)
print(f"Downloaded: {filename}")

Downloaded: test_potentialproduction_wofost72_03.yaml


In [4]:
# ---- Check the path to the files that are downloaded as explained above ----
test_data_path = "test_potentialproduction_wofost72_03.yaml"

In [5]:
# ---- Here we read the test data and set some variables ----
test_data = get_test_data(test_data_path)

crop_model_params = [
    # Leaf dynamics
    "SPAN",
    "TDWI",
    "TBASE",
    "PERDL",
    "RGRLAI",
    "KDIFTB",
    "SLATB",
    # Phenology
    "TSUMEM",
    "TBASEM",
    "TEFFMX",
    "TSUM1",
    "TSUM2",
    "DLO",
    "DLC",
    "DVSI",
    "DVSEND",
    "DTSMTB",
    # Assimilation (KDIFTB already included above)
    "AMAXTB",
    "EFFTB",
    "TMPFTB",
    "TMNFTB",
    # Respiration
    "Q10",
    "RMR",
    "RML",
    "RMS",
    "RMO",
    "RFSETB",
    # Evapotranspiration (KDIFTB already included, IAIRDU already in root)
    "CFET",
    "DEPNR",
    "IAIRDU",
    "IOX",
    "CRAIRC",
    "SM0",
    "SMW",
    "SMFCF",
    # Root dynamics (TDWI already included)
    "RDI",
    "RRI",
    "RDMCR",
    "RDMSOL",
    "RDRRTB",
    # Stem dynamics (TDWI already included)
    "RDRSTB",
    "SSATB",
    # Storage organ dynamics
    "SPA",
    # Partitioning
    "FRTB",
    "FLTB",
    "FSTB",
    "FOTB",
    # Wofost72 top-level conversion factors
    "CVL",
    "CVO",
    "CVR",
    "CVS",
]
(crop_model_params_provider, weather_data_provider, agro_management_inputs, _) = (
    prepare_engine_input(test_data, crop_model_params)
)

expected_results = test_data["ModelResults"]
expected_lai = torch.tensor(
    [float(item["LAI"]) for item in expected_results],
    dtype=ComputeConfig.get_dtype(),
    device=ComputeConfig.get_device(),
) # shape: [time_steps]

# ---- don't change this: in this config file we specify the differentiable version of Wofost72_PP ----
wofost72_config = Configuration(
    CROP=Wofost72,
    SOIL=WaterbalancePP,
    OUTPUT_VARS=["DVS", "LAI", "RD", "TAGP", "TRA", "TWLV", "TWRT", "TWSO", "TWST"],
)

### 1.3. Helper classes/functions

The model parameters should stay in a valid range. To ensure this, we will use
`BoundedParameter` class with (min, max) and initial values for each
parameter. You may change these values depending on the crop type and
location. But don't use a very small range, otherwise gradients will be very
small and the optimization will be very slow.

In [6]:
# ---- Adjust the values if needed  ----
TDWI_MIN, TDWI_MAX, TDWI_INIT = (0.0, 1.0, 0.40)
SPAN_MIN, SPAN_MAX, SPAN_INIT = (10.0, 60.0, 25.0)

# ---- Helper for bounded parameters ----
class BoundedParameter(torch.nn.Module):
    def __init__(self, low, high, init_value):
        super().__init__()
        self.low = low
        self.high = high

        # Normalize to [0, 1]
        init_norm = (init_value - low) / (high - low)

        # Parameter in raw logit space
        self.raw = torch.nn.Parameter(
            torch.logit(
                torch.tensor(
                    init_norm, dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device()
                ),
                eps=1e-6,
            )
        )

    def forward(self):
        return self.low + (self.high - self.low) * torch.sigmoid(self.raw)

Another helper class is `OptDiffWofost72PP` which is a subclass of `torch.nn.Module`. 
We use this class to wrap the `EngineTestHelper` function and make it easier to run the model `Wofost72PP`.

In [7]:
# ---- Wrap the model with torch.nn.Module----
class OptDiffWofost72PP(torch.nn.Module):
    def __init__(self, crop_model_params_provider, weather_data_provider, agro_management_inputs, wofost72_config):
        super().__init__()
        self.crop_model_params_provider = crop_model_params_provider
        self.weather_data_provider = weather_data_provider
        self.agro_management_inputs = agro_management_inputs
        self.config = wofost72_config

        # bounded parameters
        self.tdwi = BoundedParameter(TDWI_MIN, TDWI_MAX, init_value=TDWI_INIT)
        self.span = BoundedParameter(SPAN_MIN, SPAN_MAX, init_value=SPAN_INIT)

    def forward(self):
        # currently, copying is needed due to an internal issue in engine
        crop_model_params_provider_ = copy.deepcopy(self.crop_model_params_provider)

        tdwi_val = self.tdwi()
        span_val = self.span()
         
        # pass new value of parameters to the model
        crop_model_params_provider_.set_override("TDWI", tdwi_val, check=False)
        crop_model_params_provider_.set_override("SPAN", span_val, check=False)

        engine = EngineTestHelper(
            crop_model_params_provider_,
            self.weather_data_provider,
            self.agro_management_inputs,
            self.config,
        )
        engine.run_till_terminate()
        results = engine.get_output()
        
        return torch.stack([item["LAI"] for item in results]) # shape: [1, time_steps]

In [8]:
# ----  Create model ---- 
opt_model = OptDiffWofost72PP(
    crop_model_params_provider,
    weather_data_provider,
    agro_management_inputs,
    wofost72_config,
)

In [9]:
# ----  Early stopping ---- 
best_loss = float("inf")
patience = 10  # Number of steps to wait for improvement
patience_counter = 0
min_delta = 1e-4 

# ----  Optimizer ---- 
optimizer = torch.optim.Adam(opt_model.parameters(), lr=0.1)

# ----  We use relative MAE as loss because there are two outputs with different untis ----  
denom = torch.mean(torch.abs(expected_lai)) 

# Training loop (example)
for step in range(101):
    optimizer.zero_grad()
    results = opt_model()
    mae = torch.mean(torch.abs(results - expected_lai))
    rmae = mae / denom
    loss = rmae.sum()  # example: relative mean absolute error
    loss.backward()
    optimizer.step()

    print(f"Step {step}, Loss {loss.item():.4f}, TDWI {opt_model.tdwi().item():.4f}, SPAN {opt_model.span().item():.4f}")
    # Early stopping logic
    if loss.item() < best_loss - min_delta:
        best_loss = loss.item()
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at step {step}")
            break

Step 0, Loss 0.3535, TDWI 0.4242, SPAN 26.0705
Step 1, Loss 0.3122, TDWI 0.4488, SPAN 27.1814
Step 2, Loss 0.2702, TDWI 0.4736, SPAN 28.3302
Step 3, Loss 0.2242, TDWI 0.4986, SPAN 29.5137
Step 4, Loss 0.1764, TDWI 0.5235, SPAN 30.7283
Step 5, Loss 0.1390, TDWI 0.5410, SPAN 31.5807
Step 6, Loss 0.1208, TDWI 0.5524, SPAN 32.1013
Step 7, Loss 0.1071, TDWI 0.5588, SPAN 32.3388
Step 8, Loss 0.1023, TDWI 0.5612, SPAN 32.3524
Step 9, Loss 0.1024, TDWI 0.5604, SPAN 32.1868
Step 10, Loss 0.1070, TDWI 0.5569, SPAN 31.8764
Step 11, Loss 0.1165, TDWI 0.5511, SPAN 31.4542
Step 12, Loss 0.1260, TDWI 0.5434, SPAN 30.9458
Step 13, Loss 0.1388, TDWI 0.5341, SPAN 30.3713
Step 14, Loss 0.1523, TDWI 0.5234, SPAN 29.7492
Step 15, Loss 0.1661, TDWI 0.5117, SPAN 29.1053
Step 16, Loss 0.1832, TDWI 0.5008, SPAN 28.5311
Step 17, Loss 0.2047, TDWI 0.4951, SPAN 28.2217
Step 18, Loss 0.2152, TDWI 0.4938, SPAN 28.1236
Early stopping at step 18


In [10]:
# ---- validate the results using test data ---- 
print(f"Actual TDWI {crop_model_params_provider["TDWI"].item():.4f}, SPAN {crop_model_params_provider["SPAN"].item():.4f}")

Actual TDWI 0.5100, SPAN 35.0000
